# Official CODI KV subspace analysis

## Goal

Test whether the explicit-CoT teacher KV trajectory and the six latent student positions of the accuracy-gated public CODI GPT-2 checkpoint contain stable, example-paired low-rank correspondence. The model is never trained or updated here. R-KV is used only to select and chronologically align six teacher trace tokens with six student latent positions.

## 1. Choose the run scope

Run the audit first. The 2,000-example run is the preregistered primary diagnostic. Enable the independent 5,000-example seed-1 confirmation only after inspecting the primary result. Closing the browser can disconnect Colab, so keep the runtime connected while a cell is active. Intermediate moment statistics are saved atomically to Drive and the same cell can be rerun after a disconnect.

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Replace with the pushed immutable commit before the final run.
REPO_DIR = "/content/latent-reasoning"
DRIVE_ROOT = "/content/drive/MyDrive/CODI_KAVA"

RUN_ALIGNMENT_AUDIT = True
RUN_PRIMARY_2000 = True
RUN_CONFIRMATION_5000 = False

BATCH_SIZE = 16
SHUFFLE_REPEATS = 4
SAVE_EVERY = 500
PRECISION = "bfloat16"


## 2. Mount Drive and install the pinned environment

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pathlib
import subprocess
import sys

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

repo = pathlib.Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
target = f"origin/{RUN_COMMIT}" if RUN_COMMIT == "main" else RUN_COMMIT
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", target], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-official-codi.txt")],
    check=True,
)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main":
    print("Pin RUN_COMMIT before the final collection to:", commit)


## 3. Verify the GPU, code contracts, and passed accuracy gate

In [ ]:
import json
import torch

assert torch.cuda.is_available(), "Enable a Colab GPU runtime"
gpu_name = torch.cuda.get_device_name(0)
print("Torch:", torch.__version__)
print("GPU:", gpu_name)
if "A100" not in gpu_name:
    print("Warning: the workflow will run on this GPU, but the time estimate assumed an A100.")

subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_official_codi.py",
        "tests/test_official_codi_kv.py",
        "tests/test_kv_compress.py",
        "tests/test_kv_cross_subspace.py",
        "tests/test_kv_reduced_rank.py",
    ],
    cwd=REPO_DIR,
    check=True,
)

official_root = pathlib.Path(DRIVE_ROOT) / "outputs" / "official_codi_gpt2"
gate_candidates = sorted(official_root.glob("eval/revision_*/full_gsm8k/summary.json"))
passed = []
for candidate in gate_candidates:
    payload = json.loads(candidate.read_text())
    gate = payload.get("accuracy_gate", payload.get("gate"))
    status = gate if isinstance(gate, str) else (gate or {}).get("status")
    if status == "passed":
        passed.append(candidate)
assert passed, "Run colab_official_codi_validation.ipynb until full GSM8K passes"
REPRODUCTION_SUMMARY = passed[-1]
print("Passed reproduction summary:", REPRODUCTION_SUMMARY)
print(json.dumps(json.loads(REPRODUCTION_SUMMARY.read_text()), indent=2))


## 4. Define Drive-persistent execution

In [ ]:
import datetime

OUTPUT_ROOT = pathlib.Path(DRIVE_ROOT) / "outputs" / "official_codi_kv_subspaces"
REPORT_ROOT = pathlib.Path(DRIVE_ROOT) / "reports" / "official_codi_kv_subspaces"
LOG_ROOT = pathlib.Path(DRIVE_ROOT) / "logs" / "official_codi_kv_subspaces"
for path in (OUTPUT_ROOT, REPORT_ROOT, LOG_ROOT):
    path.mkdir(parents=True, exist_ok=True)

def run_persisted(cmd, log_name):
    log_path = LOG_ROOT / log_name
    print("Starting:", " ".join(map(str, cmd)), flush=True)
    print("Persistent log:", log_path, flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} {' '.join(map(str, cmd))} ===\n")
        process = subprocess.Popen(
            list(map(str, cmd)),
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
        log.flush()
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)
    return code

def collection_command(output_dir, examples, seed, audit_only=False):
    cmd = [
        sys.executable, "-u", "scripts/collect_official_codi_kv_subspaces.py",
        "--config", "configs/official_codi_gpt2.yaml",
        "--reproduction-summary", str(REPRODUCTION_SUMMARY),
        "--output-dir", str(output_dir),
        "--examples", str(examples),
        "--batch-size", str(BATCH_SIZE),
        "--shuffle-repeats", str(SHUFFLE_REPEATS),
        "--save-every", str(SAVE_EVERY),
        "--precision", PRECISION,
        "--device", "cuda",
        "--seed", str(seed),
    ]
    if audit_only:
        cmd.append("--audit-only")
    return cmd

def analyze_collection(statistics_dir, report_stem):
    state = torch.load(statistics_dir / "statistics.pt", map_location="cpu", weights_only=False)
    position_counts = state["moments"]["actual"]["key"]["position_count"]
    minimum_counts = position_counts.amin(dim=(0, 1, 2))
    print("Minimum observations per split/layer/head by latent position:", minimum_counts.tolist())
    assert int(minimum_counts.min()) >= 128, "Insufficient trace coverage for a stable 64-dimensional split analysis"
    del state, position_counts
    stage1b = REPORT_ROOT / f"{report_stem}_cross_subspace.json"
    stage1c = REPORT_ROOT / f"{report_stem}_reduced_rank.json"
    run_persisted(
        [sys.executable, "scripts/analyze_kv_cross_subspaces.py", "--statistics", str(statistics_dir), "--output", str(stage1b)],
        f"{report_stem}_analyze_stage1b.log",
    )
    run_persisted(
        [sys.executable, "scripts/analyze_kv_reduced_rank.py", "--statistics", str(statistics_dir), "--output", str(stage1c)],
        f"{report_stem}_analyze_stage1c.log",
    )
    return stage1b, stage1c


## 5. Run the one-batch alignment audit

This validates teacher and student padding, trace boundaries, R-KV indices, the 12-layer by 12-head by 6-position by 64-dimension KV contract, and finite values. It deliberately does not allocate or save the roughly 400 MB moment collection.

In [ ]:
from IPython.display import JSON, Markdown, display

AUDIT_DIR = OUTPUT_ROOT / "audit_seed0"
if RUN_ALIGNMENT_AUDIT:
    run_persisted(
        collection_command(AUDIT_DIR, BATCH_SIZE, 0, audit_only=True),
        "alignment_audit.log",
    )
    audit = json.loads((AUDIT_DIR / "alignment_audit.json").read_text())
    assert audit["teacher_selected_shape"] == audit["student_latent_shape"]
    assert audit["teacher_selected_shape"][1:] == [12, 12, 6, 64]
    assert audit["selected_valid_fraction"] > 0.0, "No valid explicit-CoT teacher targets were extracted"
    assert audit["finite_teacher_keys"] and audit["finite_teacher_values"] and audit["finite_student_keys"]
    if audit["selected_valid_fraction"] < 0.90:
        print("Coverage note: some official examples have fewer than six trace targets after the released final-step removal rule.")
        print("The collector masks those missing targets; inspect per-position coverage below and the final group counts.")
    display(JSON(audit))
else:
    print("Alignment audit skipped")


## 6. Run the 2,000-example primary analysis

The collector alternates complete batches between two independent halves. It accumulates actual teacher-student cross-moments and four within-batch derangement nulls. Stage 1b measures split-stable canonical directions. Stage 1c fits reduced-rank maps on one half and evaluates them on the untouched half.

In [ ]:
PRIMARY_DIR = OUTPUT_ROOT / "n2000_seed0"
if RUN_PRIMARY_2000:
    run_persisted(
        collection_command(PRIMARY_DIR, 2000, 0),
        "collect_n2000_seed0.log",
    )
    primary_stage1b, primary_stage1c = analyze_collection(
        PRIMARY_DIR,
        "official_codi_n2000_seed0",
    )
    display(Markdown(primary_stage1b.with_suffix(".md").read_text()))
    display(Markdown(primary_stage1c.with_suffix(".md").read_text()))
else:
    print("Primary analysis skipped")


## 7. Optionally run the independent 5,000-example confirmation

This uses seed 1, not the primary seed-0 prefix. Treat it as confirmation rather than tuning data. Enable it only after recording the primary gate result.

In [ ]:
CONFIRMATION_DIR = OUTPUT_ROOT / "n5000_seed1"
if RUN_CONFIRMATION_5000:
    run_persisted(
        collection_command(CONFIRMATION_DIR, 5000, 1),
        "collect_n5000_seed1.log",
    )
    confirm_stage1b, confirm_stage1c = analyze_collection(
        CONFIRMATION_DIR,
        "official_codi_n5000_seed1",
    )
    display(Markdown(confirm_stage1b.with_suffix(".md").read_text()))
    display(Markdown(confirm_stage1c.with_suffix(".md").read_text()))
else:
    print("Confirmation skipped. Set RUN_CONFIRMATION_5000 = True after recording the primary result.")


## 8. Inspect durable artifacts and interpretation boundary

In [ ]:
print("\nDurable reports")
for path in sorted(REPORT_ROOT.glob("official_codi_*")):
    print(path.name, f"{path.stat().st_size / 1024:.1f} KiB")

print("\nDurable collections")
for path in sorted(OUTPUT_ROOT.glob("*/collection_manifest.json")):
    payload = json.loads(path.read_text())
    print(path.parent.name, payload.get("state"), payload.get("processed_examples"))

display(Markdown("""
### Interpretation boundary

A positive held-out reduced-rank gate means a small position-conditioned linear subspace predicts teacher KV information beyond shuffled pairing in a paper-accuracy CODI checkpoint. It does not show answer causality or an accuracy improvement. A downstream projection-versus-full-versus-random, compute-matched experiment would still be required. A negative result means this R-KV alignment and linear diagnostic did not isolate a stable transferable subspace.
"""))
